<a href="https://colab.research.google.com/github/sahilkh11111-ship-it/research-paper-agent-assignment1/blob/main/recipe_agent_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About
This colab is an introduction to a RAG based ReAct agent that looks at a recipe book, reasons the best course of action and generates a response.

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to aistudio.google.com
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [ ]:
# @title Creatw a database of recipes.
'''RECIPES = {
    "bombe_alaska": (
            "1 pint vanilla ice cream softened"
            "1 pint raspberry sorbet softened"
            "1 pint chocolate ice cream softened"
            "1 prepared 9-inch sponge cake or genoise layer"
            "4 large egg whites"
            "1/2 cup granulated sugar"
            "Pinch of cream of tartar"
            "Line a 2-quart bowl with plastic wrap leaving an overhang. Spoon vanilla ice cream into the bowl spreading evenly. Freeze for 30 minutes."
            "Repeat with raspberry sorbet then chocolate ice cream freezing each layer for 30 minutes."
            "Place the cake layer on top of the chocolate ice cream pressing gently. Cover with plastic wrap and freeze overnight."
            "Preheat oven to 500°F (260°C). In a large bowl whisk egg whites until foamy. Gradually add sugar and cream of tartar whisking until stiff glossy peaks form (meringue)."
            "Unmold the ice cream dome onto an oven-safe serving plate. Remove plastic wrap. Spread meringue all over the ice cream dome ensuring it's completely covered and sealed to the plate."
            "Bake for 3-5 minutes or until meringue is golden brown. Serve immediately."
            ),
    "fried_rice": (
            "3 cups cooked day-old white rice"
            "2 tbsp vegetable oil"
            "1/2 cup diced carrots"
            "1/2 cup frozen peas"
            "1/2 cup chopped onion"
            "2 cloves garlic minced"
            "2 large eggs beaten"
            "2 tbsp soy sauce"
            "1 tbsp oyster sauce (optional)"
            "1 tsp sesame oil"
            "Salt and pepper to taste"
            "Green onions chopped for garnish"
            "Heat vegetable oil in a large skillet or wok over medium-high heat."
            "Add carrots peas and onion. Cook stirring frequently for 3-5 minutes until vegetables are tender-crisp."
            "Add minced garlic and cook for another minute until fragrant."
            "Push vegetables to one side of the skillet. Pour beaten eggs into the empty side and scramble until cooked through. Break into small pieces."
            "Add cooked rice to the skillet. Pour soy sauce oyster sauce (if using) and sesame oil over the rice. Stir everything together breaking up any clumps of rice."
            "Cook for 5-7 minutes stirring occasionally until the rice is heated through and slightly browned."
            "Season with salt and pepper to taste. Garnish with chopped green onions before serving."
    ),
    "butter_chicken": (
            "1.5 lbs boneless skinless chicken thighs cut into 1-inch pieces"
            "1/2 cup plain yogurt"
            "1 tbsp lemon juice"
            "2 tsp ground cumin"
            "2 tsp garam masala"
            "1 tsp turmeric powder"
            "1 tsp chili powder (or paprika for milder taste)"
            "1/2 tsp salt"
            "2 tbsp butter"
            "1 large onion finely chopped"
            "2 cloves garlic minced"
            "1 tbsp grated fresh ginger"
            "1 (14.5 ounce) can crushed tomatoes"
            "1/2 cup heavy cream"
            "1/4 cup chopped fresh cilantro for garnish"
            "In a large bowl combine chicken pieces with yogurt lemon juice cumin garam masala turmeric chili powder and salt. Mix well to coat the chicken. Marinate for at least 30 minutes or preferably 2-4 hours in the refrigerator."
            "Melt butter in a large skillet or Dutch oven over medium heat. Add chopped onion and cook until softened and translucent about 5-7 minutes."
            "Add minced garlic and grated ginger cook for 1 minute until fragrant."
            "Add the marinated chicken to the skillet. Cook stirring occasionally until browned on all sides about 8-10 minutes."
            "Stir in the crushed tomatoes. Bring to a simmer then reduce heat to low cover and cook for 15-20 minutes or until chicken is cooked through."
            "Stir in the heavy cream. Cook for another 5 minutes uncovered until the sauce thickens slightly. Adjust seasoning if needed."
            "Serve hot over rice or with naan bread garnished with fresh cilantro."
    ),
    "alfredo_pasta": (
            "1 lb fettuccine pasta"
            "1/2 cup unsalted butter"
            "2 cups heavy cream"
            "1.5 cups freshly grated Parmesan cheese"
            "1/4 tsp ground nutmeg (optional)"
            "Salt and freshly ground black pepper to taste"
            "Fresh parsley chopped for garnish"
            "Cook fettuccine according to package directions until al dente. Reserve 1 cup of pasta water before draining."
            "While pasta cooks melt butter in a large skillet or saucepan over medium-low heat."
            "Pour in heavy cream and bring to a gentle simmer stirring occasionally. Do not boil."
            "Reduce heat to low. Gradually whisk in the Parmesan cheese until melted and the sauce is smooth. If the sauce is too thick add a little reserved pasta water until desired consistency is reached."
            "Season with nutmeg (if using) salt and freshly ground black pepper. Taste and adjust seasoning."
            "Add the drained fettuccine to the sauce and toss to coat evenly."
            "Serve immediately garnished with chopped fresh parsley."
    )
}'''

ANIME = {
    "attack_on_titan": (
        "Title: Attack on Titan"
        "Japanese Title: Shingeki no Kyojin"
        "Type: TV Series"
        "Status: Completed"
        "Episodes: 87"
        "Genres: Action, Dark Fantasy, Drama, Mystery, Military"
        "Plot: Humanity lives behind enormous walls to protect itself from giant humanoid creatures called Titans. After his hometown is destroyed and his mother is killed, Eren Yeager vows to eliminate every Titan. He joins the military alongside Mikasa Ackerman and Armin Arlert. As the story progresses, they uncover the truth about the Titans, the walls, their own world, and a much larger conflict involving humanity and the outside world."
        "Setting: A fictional world divided between nations and territories, with humanity initially living inside three enormous walls."
        "Main Character: Eren Yeager"
        "Main Character Type: Protagonist, Anti-Hero, Titan Shifter"
        "Eren Personality: Determined, impulsive, passionate, rebellious, protective"
        "Eren Abilities: Titan transformation, regeneration, hand-to-hand combat, strategic thinking"
        "Other Main Characters: Mikasa Ackerman, Armin Arlert, Levi Ackerman, Erwin Smith"
        "Mikasa Character Type: Warrior, Soldier, Protector"
        "Armin Character Type: Strategist, Leader, Titan Shifter"
        "Levi Character Type: Elite Soldier, Captain"
        "Power System: Titan powers, ODM gear, military combat"
        "Themes: Freedom, war, sacrifice, revenge, morality, survival, political conflict, humanity"
        "Tone: Dark, serious, emotional, violent, psychological"
        "Character Types: Protagonist, Anti-Hero, Warrior, Strategist, Soldier, Titan Shifter, Commander, Antagonist"
        "Target Audience: Shonen"
    ),

    "naruto": (
        "Title: Naruto"
        "Japanese Title: Naruto"
        "Type: TV Series"
        "Status: Completed"
        "Episodes: 220"
        "Genres: Action, Adventure, Martial Arts, Fantasy, Shonen"
        "Plot: Naruto Uzumaki is a young ninja from the Hidden Leaf Village who dreams of becoming Hokage, the strongest and most respected ninja in his village. Because the Nine-Tailed Fox was sealed inside him when he was a baby, Naruto grows up isolated and rejected by many people. Through training, friendship, rivalry, and battles, Naruto gradually earns the respect of others while discovering the history of the ninja world and the secrets surrounding his own identity."
        "Setting: A fictional world consisting of several ninja villages, each with its own leaders, clans, traditions, and military forces."
        "Main Character: Naruto Uzumaki"
        "Main Character Type: Protagonist, Shonen Hero, Jinchuriki"
        "Naruto Personality: Energetic, optimistic, stubborn, loyal, determined, humorous"
        "Naruto Abilities: Shadow Clone Jutsu, Rasengan, Sage Mode, Nine-Tails Chakra, powerful chakra control"
        "Other Main Characters: Sasuke Uchiha, Sakura Haruno, Kakashi Hatake"
        "Sasuke Character Type: Rival, Anti-Hero, Uchiha"
        "Sakura Character Type: Medic Ninja, Fighter"
        "Kakashi Character Type: Sensei, Elite Ninja, Strategist"
        "Power System: Chakra, Ninjutsu, Genjutsu, Taijutsu, Kekkei Genkai"
        "Themes: Friendship, loneliness, perseverance, revenge, identity, family, acceptance, determination"
        "Tone: Adventurous, emotional, comedic, dramatic, action-packed"
        "Character Types: Protagonist, Rival, Sensei, Villain, Anti-Hero, Medic Ninja, Strategist, Jinchuriki"
        "Target Audience: Shonen"
    ),

    "death_note": (
        "Title: Death Note"
        "Japanese Title: Desu Noto"
        "Type: TV Series"
        "Status: Completed"
        "Episodes: 37"
        "Genres: Psychological Thriller, Mystery, Supernatural, Crime, Drama"
        "Plot: Light Yagami, an exceptionally intelligent student, discovers a mysterious notebook called the Death Note. Anyone whose name is written in the notebook will die, provided the user knows their face. Light begins using the notebook to eliminate criminals and create a world without crime. His actions attract the attention of the mysterious detective L, leading to an intense psychological battle between the two."
        "Setting: Modern-day Japan with supernatural elements involving mysterious Shinigami and the Death Note."
        "Main Character: Light Yagami"
        "Main Character Type: Protagonist, Anti-Villain, Genius"
        "Light Personality: Intelligent, calculating, ambitious, manipulative, confident"
        "Light Abilities: Exceptional intelligence, manipulation, strategic planning, use of the Death Note"
        "Other Main Characters: L, Ryuk, Misa Amane, Near"
        "L Character Type: Detective, Rival, Genius"
        "Ryuk Character Type: Shinigami, Supernatural Observer"
        "Misa Character Type: Kira Supporter, Idol, Devoted Ally"
        "Power System: Death Note, Shinigami, supernatural rules"
        "Themes: Justice, morality, power, corruption, good versus evil, human nature, crime"
        "Tone: Dark, psychological, suspenseful, mysterious, serious"
        "Character Types: Genius, Detective, Anti-Villain, Shinigami, Investigator, Criminal, Rival"
        "Target Audience: Shonen"
    ),

    "one_piece": (
        "Title: One Piece"
        "Japanese Title: One Piece"
        "Type: TV Series"
        "Status: Ongoing"
        "Episodes: 1000+"
        "Genres: Action, Adventure, Fantasy, Comedy, Shonen"
        "Plot: Monkey D. Luffy is a young pirate who dreams of becoming the Pirate King. He sets out on a journey across the oceans to find the legendary treasure known as the One Piece. Along the way, he forms the Straw Hat Pirates and gathers companions with their own dreams and ambitions. Their journey leads them into conflicts with pirates, Marines, powerful organizations, and the World Government while gradually revealing the mysteries of the world."
        "Setting: A vast fictional world consisting of oceans, islands, pirate territories, kingdoms, and powerful global organizations."
        "Main Character: Monkey D. Luffy"
        "Main Character Type: Protagonist, Pirate Captain, Shonen Hero"
        "Luffy Personality: Cheerful, fearless, loyal, adventurous, impulsive, determined"
        "Luffy Abilities: Devil Fruit powers, Haki, enhanced physical abilities, advanced combat"
        "Other Main Characters: Roronoa Zoro, Nami, Sanji, Usopp, Tony Tony Chopper"
        "Zoro Character Type: Swordsman, Combatant, First Mate"
        "Nami Character Type: Navigator, Strategist"
        "Sanji Character Type: Fighter, Cook"
        "Chopper Character Type: Doctor, Devil Fruit User"
        "Power System: Devil Fruits, Haki, martial arts, swordsmanship"
        "Themes: Freedom, friendship, dreams, adventure, justice, oppression, family, loyalty"
        "Tone: Adventurous, funny, emotional, dramatic, epic"
        "Character Types: Pirate, Captain, Swordsman, Navigator, Doctor, Fighter, Marine, Villain, Revolutionary"
        "Target Audience: Shonen"
    ),

    "demon_slayer": (
        "Title: Demon Slayer"
        "Japanese Title: Kimetsu no Yaiba"
        "Type: TV Series"
        "Status: Ongoing"
        "Episodes: Multiple Seasons"
        "Genres: Action, Dark Fantasy, Adventure, Supernatural, Shonen"
        "Plot: Tanjiro Kamado is a kind-hearted boy whose family is slaughtered by demons. His younger sister Nezuko survives but is transformed into a demon. Determined to find a way to turn Nezuko back into a human and defeat the demon responsible for his family's deaths, Tanjiro joins the Demon Slayer Corps. He trains alongside powerful swordsmen and faces increasingly dangerous demons."
        "Setting: Historical Japan during the Taisho era, where demons secretly prey upon humans."
        "Main Character: Tanjiro Kamado"
        "Main Character Type: Protagonist, Demon Slayer"
        "Tanjiro Personality: Kind, compassionate, determined, courageous, responsible"
        "Tanjiro Abilities: Water Breathing, Hinokami Kagura, swordsmanship, enhanced senses"
        "Other Main Characters: Nezuko Kamado, Zenitsu Agatsuma, Inosuke Hashibira, Giyu Tomioka"
        "Nezuko Character Type: Demon, Protector, Supporting Character"
        "Zenitsu Character Type: Swordsman, Comedic Character, Thunder Breathing User"
        "Inosuke Character Type: Warrior, Beast-like Fighter, Swordsman"
        "Giyu Character Type: Hashira, Elite Demon Slayer"
        "Power System: Breathing Styles, Demon Blood Arts, Demon transformations"
        "Themes: Family, sacrifice, compassion, perseverance, revenge, humanity"
        "Tone: Emotional, dark, dramatic, action-packed, occasionally comedic"
        "Character Types: Demon Slayer, Demon, Hashira, Protagonist, Swordsman, Warrior, Mentor, Villain"
        "Target Audience: Shonen"
    )
}

In [ ]:
# @title Define a tool
def search_animes(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in ANIME.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Quick test
print(search_animes("deathnote"))


No relevant recipes found for that query.


In [ ]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_animes],
    system_instruction=(
        '''
        you are a anime expert , who has all the knowledge about plot info
        story character and can derive the meaning behind the story , you are
        always thorough with the word keeping it concise yet packing all the details
        in it
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [ ]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_anime",
        description="Search the anime to find most the relevant info  to give detail.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_animes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("who is light yagami")

Question: who is light yagami
--------------------------------------------------

Final answer:
**Light Yagami** (夜神 月, *Yagami Raito*) is the main protagonist (and anti-hero/villain) of the critically acclaimed anime and manga series ***Death Note***, created by Tsugumi Ohba and Takeshi Obata. 

Here is a breakdown of who he is, his role in the story, and his psychological transformation:

---

### 1. Basic Profile
* **Role:** High school student turned serial killer / self-proclaimed "God of the New World."
* **Alias:** **Kira** (キラ), derived from the Japanese pronunciation of the English word "Killer."
* **Goal:** To cleanse the world of criminals and evil people to create a peaceful utopia where he acts as the supreme ruler/god.

### 2. Personality & Intelligence
* **Genius:** Light is introduced as a bored, incredibly gifted high school student who consistently ranks as the top academic student in all of Japan. He excels at strategy, academics, athletics, and manipulation.
* **Arr

In [ ]:
questions = [
    "What is the main plot of Attack on Titan?",
    "Who is the main character in Naruto?",
    "What type of character is Light Yagami?",
    "What abilities does Monkey D. Luffy have?",
    "What are the main themes of Demon Slayer?",
    "Which anime has a character called Ryuk?",
    "Which anime features Titans as the main threat?",
    "What is the target audience of One Piece?",
    "Who are the main characters in Attack on Titan?",
    "What power system is used in Naruto?",

    # These require information that is not directly available
    "How many people watched Attack on Titan worldwide?",
    "What is the total production budget of One Piece?",
    "How many anime fans prefer Naruto over One Piece?",
    "How many hours would it take to watch all episodes of One Piece?",

    # General knowledge questions not answered by our database
    "What is the most popular anime in Japan?",
    "Who created Dragon Ball?",
    "What is the highest-rated anime on IMDb?",
    "When was the first anime created?",
    "What is the most famous anime character of all time?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What is the main plot of Attack on Titan?
--------------------------------------------------
Step 1: model calls search_anime({'query': 'Attack on Titan plot summary'})
  Result preview: [Source: attack_on_titan]
Title: Attack on TitanJapanese Title: Shingeki no KyojinType: TV SeriesSta...

Final answer:
At its core, **Attack on Titan** (*Shingeki no Kyojin*) is a dark fantasy, mystery, and military drama that evolves from a localized survival story into a massive, complex geopolitical conflict. 

The main plot can be broken down into distinct narrative stages:

### 1. The Survival Stage (The Fall of Humanity)
For a century, the remnants of humanity have lived inside a civilization surrounded by three massive concentric walls (**Wall Maria, Wall Rose, and Wall Sina**) to protect themselves from giant, mindless humanoid creatures known as **Titans**, who devour humans instinctively. 

The story follows **Eren Yeager**, his adoptive sister **Mikasa Ackerman**, and their childh

# Next steps:
* Save a copy of this code to your GitHub repo.
* Create a tool to recommend ingredient substitutions for recipes.
* Push the new tool to your colab and v2 to the same GitHub repo.